<a href="https://colab.research.google.com/github/Malaika-05/ML_mini-Tasks/blob/main/Customer_Churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Prediction
---



In [ ]:
!pip install scikit-learn pandas numpy joblib imblearn -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings
warnings.filterwarnings("ignore")


1. Load Data (Pandas)
2. Preprocess (Scaler + Encoder)
3. Split Data (train/test)
4. Train Model (Logistic / RandomForest)
5. Optimize (GridSearchCV)
6. Evaluate (Accuracy, F1)
7. Save Model (Joblib)

#Step 2: Load the Telco Churn Dataset

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Drop customerID - it's just an identifier, not a feature
df.drop(columns=["customerID"], inplace=True)  #Feature Selection

# TotalCharges has hidden spaces - fix and convert to float
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)

# Convert target: Yes → 1, No → 0
df["Churn"] = (df["Churn"] == "Yes").astype(int)

print("Shape:", df.shape)
print("\nChurn distribution:")
print(df["Churn"].value_counts())

Shape: (7032, 20)

Churn distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


Converts TotalCharges column to numeric (float)

errors="coerce" → invalid values become NaN



---

This code segment performs initial data preprocessing for the Telco Customer Churn dataset. It begins by loading the dataset from an external URL into a Pandas DataFrame. The customerID column is removed as it does not contribute to prediction. The TotalCharges column is cleaned and converted into a numeric format, with invalid entries handled by converting them to missing values and subsequently removing those rows.

The target variable Churn is transformed from categorical values ("Yes"/"No") into binary numerical format (1/0) to make it suitable for machine learning models. Finally, the dataset’s shape and class distribution are displayed to understand its structure and detect any imbalance in the target variable.



#Step 3: Split Features & Handle Class Imbalance with SMOTE

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Identify column types
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [ ]:
# First encode categoricals so SMOTE can work (SMOTE needs numeric input)
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Train/test split BEFORE applying SMOTE (never oversample test data)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE only on training data to fix imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nBefore SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE :", pd.Series(y_train_balanced).value_counts().to_dict())


Before SMOTE: {0: 4130, 1: 1495}
After SMOTE : {0: 4130, 1: 4130}


SMOTE = Synthetic Minority Oversampling Technique

**What SMOTE Does**

Instead of copying data, it:

Creates synthetic (new) samples

Uses nearest neighbors

Generates realistic new points


---

This code segment prepares the dataset for machine learning by separating features and the target variable, identifying categorical and numerical columns, and converting categorical variables into numeric format using one-hot encoding. The dataset is then split into training and testing sets using stratified sampling to preserve class distribution.

To address class imbalance in the training data, SMOTE (Synthetic Minority Oversampling Technique) is applied, which generates synthetic samples for the minority class instead of duplicating existing data. This results in a balanced training dataset, improving the model’s ability to learn patterns from both classes effectively. The class distribution before and after applying SMOTE is displayed to verify the balancing process.



#Step 4: Build the Sklearn Pipeline

In [ ]:
# All columns are now numeric after get_dummies, so just scale them
preprocessor = StandardScaler()

# Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
# Random Forest Pipeline
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

print("Pipelines created.")

Pipelines created.


This code defines preprocessing and modeling pipelines using scikit-learn. A StandardScaler is used to normalize all numerical features, ensuring consistent feature scaling across the dataset. Two separate pipelines are created: one for Logistic Regression and another for Random Forest classification.

Each pipeline integrates data scaling and model training into a single workflow, improving code organization and reducing the risk of preprocessing errors. Logistic Regression is configured with increased iterations to ensure convergence, while Random Forest provides a robust ensemble-based alternative. These pipelines allow seamless training and evaluation without manually applying preprocessing steps.

#Step 5: Hyperparameter Tuning with GridSearchCV

In [ ]:
# --- Logistic Regression Grid Search ---
lr_params = {
    "model__C": [0.01, 0.1, 1, 10],           # Regularization strength
    "model__solver": ["lbfgs", "liblinear"]    # Optimization algorithm
}

lr_grid = GridSearchCV(
    lr_pipeline,
    lr_params,
    cv=5,              # 5-fold cross validation
    scoring="f1",      # Optimize for F1 since data was imbalanced
    n_jobs=-1,         # Use all CPU cores
    verbose=1
)

lr_grid.fit(X_train_balanced, y_train_balanced)
print("\nBest LR params:", lr_grid.best_params_)
print(f"Best LR CV F1 : {lr_grid.best_score_:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best LR params: {'model__C': 1, 'model__solver': 'lbfgs'}
Best LR CV F1 : 0.8150


In [ ]:
# --- Random Forest Grid Search ---
rf_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_balanced, y_train_balanced)
print("\nBest RF params:", rf_grid.best_params_)
print("Best RF CV F1 :", f"{rf_grid.best_score_:.4f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best RF params: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best RF CV F1 : 0.8407


**What is Hyperparameter Tuning?**

✔ Definition

Hyperparameters are settings you choose before training a model.

**Examples:**

Learning rate

Number of trees

Regularization strength

👉 These are not learned by the model—they are controlled by you.


---
**C (Regularization Strength)**

Controls how much the model avoids overfitting

Value	Effect

Small (0.01)	Strong regularization (simpler model)

Large (10)	Weak regularization (complex model)



---





**solver (Optimization Algorithm)**

Method used to train the model

Examples:

"lbfgs" → fast, good for large datasets

"liblinear" → good for smaller datasets


---

| Term             | Meaning                            |
| ---------------- | ---------------------------------- |
| Hyperparameter   | Setting you choose before training |
| GridSearchCV     | Tries all parameter combinations   |
| Cross-validation | Repeated training for reliability  |
| F1-score         | Balance between precision & recall |
| Regularization   | Prevents overfitting               |



---

This section performs hyperparameter tuning using GridSearchCV to optimize both Logistic Regression and Random Forest models. A grid of possible parameter values is defined for each model, and all combinations are evaluated using 5-fold cross-validation. The models are optimized based on the F1-score, which is particularly suitable for imbalanced datasets.

For Logistic Regression, parameters such as regularization strength (C) and solver type are tuned. For Random Forest, the number of trees, maximum tree depth, and minimum samples required for splitting are optimized. The grid search process evaluates multiple model configurations and selects the best-performing parameters based on cross-validation results. The final output shows the optimal parameter settings and their corresponding F1-scores, with Random Forest achieving better performance than Logistic Regression




#Step 6: Evaluate Both Models on Test Data

In [ ]:
def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

evaluate("Logistic Regression", lr_grid.best_estimator_, X_test, y_test)
evaluate("Random Forest",       rf_grid.best_estimator_, X_test, y_test)


  Logistic Regression
              precision    recall  f1-score   support

    No Churn       0.86      0.81      0.84      1033
       Churn       0.55      0.65      0.60       374

    accuracy                           0.77      1407
   macro avg       0.71      0.73      0.72      1407
weighted avg       0.78      0.77      0.77      1407


  Random Forest
              precision    recall  f1-score   support

    No Churn       0.86      0.84      0.85      1033
       Churn       0.58      0.61      0.59       374

    accuracy                           0.78      1407
   macro avg       0.72      0.72      0.72      1407
weighted avg       0.78      0.78      0.78      1407



This section evaluates the performance of the trained Logistic Regression and Random Forest models on the test dataset. A reusable evaluation function is defined to generate predictions and display a classification report for each model. The report includes precision, recall, F1-score, accuracy, and support for both churn and non-churn classes.

The results show that Random Forest slightly outperforms Logistic Regression in terms of overall accuracy and F1-score. Both models perform well in predicting the majority class (non-churn), but show comparatively lower performance for the minority class (churn), highlighting the challenges of class imbalance. The use of F1-score provides a more balanced evaluation metric, ensuring that both precision and recall are considered when assessing model performance.

#Step 7: Pick Best Model & Export with Joblib

In [ ]:
# Compare test F1 scores
from sklearn.metrics import f1_score

lr_f1 = f1_score(y_test, lr_grid.best_estimator_.predict(X_test))
rf_f1 = f1_score(y_test, rf_grid.best_estimator_.predict(X_test))

print(f"LR Test F1: {lr_f1:.4f}")
print(f"RF Test F1: {rf_f1:.4f}")

# Save the better model
best_model = lr_grid.best_estimator_ if lr_f1 > rf_f1 else rf_grid.best_estimator_
best_name  = "Logistic Regression" if lr_f1 > rf_f1 else "Random Forest"

joblib.dump(best_model, "churn_pipeline.pkl")
print(f"\n✅ Saved: {best_name} → churn_pipeline.pkl")

LR Test F1: 0.5953
RF Test F1: 0.5919

✅ Saved: Logistic Regression → churn_pipeline.pkl


#Step 8: Load & Use the Saved Pipeline

In [ ]:
# Simulate production use — load and predict on new data
loaded_pipeline = joblib.load("churn_pipeline.pkl")

# Use one test sample as a "new customer"
sample = X_test.iloc[[0]]
prediction = loaded_pipeline.predict(sample)[0]
probability = loaded_pipeline.predict_proba(sample)[0][1]

print(f"\nPrediction : {'Churn' if prediction == 1 else 'No Churn'}")
print(f"Churn Probability: {probability*100:.1f}%")


Prediction : No Churn
Churn Probability: 1.9%
